In [ ]:
import importlib
import json
import math
import subprocess
import sys
import time
import traceback
import warnings
from collections import deque
from pathlib import Path

OPTIONAL_PACKAGES = {
    "mediapipe": "mediapipe",
    "gradio": "gradio>=4.44.0",
}
OPTIONAL_STATUS = {}
for module_name, package_spec in OPTIONAL_PACKAGES.items():
    try:
        importlib.import_module(module_name)
        OPTIONAL_STATUS[module_name] = True
    except Exception:
        try:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", package_spec], check=True)
            importlib.import_module(module_name)
            OPTIONAL_STATUS[module_name] = True
        except Exception as exc:
            OPTIONAL_STATUS[module_name] = False
            print(f"{module_name} install skipped: {exc}")

import cv2
import numpy as np
import pandas as pd
from PIL import Image

try:
    from IPython.display import Video, clear_output, display
except Exception:
    Video = None

    def clear_output(wait=False):
        return None

    def display(obj):
        return None

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import mobilenet_v2

warnings.filterwarnings("ignore")
torch.backends.cudnn.benchmark = True

CLASS_NAMES = ["Cover Drive", "Pull Shot", "Straight Drive", "Cut Shot", "Sweep", "Defence"]
IMG_SIZE = 224
SEQ_LEN = 16
INFER_EVERY = 5
SMOOTH_WINDOW = 5
CONF_THRESH = 0.60
LOG_COOLDOWN = 1.50
INLINE_DISPLAY_EVERY = 2
WINDOW_NAME = "Real-Time Cricket Shot Detection"

KAGGLE_ENV = Path("/kaggle").exists()
WORK_DIR = Path("/kaggle/working") if KAGGLE_ENV else Path.cwd()
WORK_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_VIDEO_PATH = str(WORK_DIR / "realtime_shot_detection_output.mp4")
TIMESTAMP_CSV_PATH = str(WORK_DIR / "shot_timestamps.csv")
TIMESTAMP_JSON_PATH = str(WORK_DIR / "shot_timestamps.json")
DEMO_VIDEO_PATH = str(WORK_DIR / "demo_cricket_stream.mp4")

MODEL_PATH = None
VIDEO_PATH = None
VIDEO_SOURCE = 0
USE_POSE = False
SHOW_INLINE = KAGGLE_ENV
SIMULATE_REALTIME_FOR_FILE = True
MAX_FRAMES = None
LAUNCH_GRADIO = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

try:
    import mediapipe as mp
    MEDIAPIPE_AVAILABLE = True
except Exception:
    mp = None
    MEDIAPIPE_AVAILABLE = False

try:
    import gradio as gr
    GRADIO_AVAILABLE = True
except Exception:
    gr = None
    GRADIO_AVAILABLE = False

print(json.dumps({
    "device": str(DEVICE),
    "kaggle_env": KAGGLE_ENV,
    "mediapipe_available": MEDIAPIPE_AVAILABLE,
    "gradio_available": GRADIO_AVAILABLE,
    "work_dir": str(WORK_DIR),
}, indent=2))


In [ ]:
class MobileNetV2LSTM(nn.Module):
    def __init__(self, num_classes=6, hidden_dim=256, dropout=0.25, use_pose=False, pose_dim=132):
        super().__init__()
        backbone = mobilenet_v2(weights=None)
        self.backbone = backbone.features
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.feature_proj = nn.Sequential(
            nn.Linear(1280, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
        )
        self.temporal = nn.LSTM(
            input_size=256,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
        )
        self.use_pose = use_pose
        self.pose_dim = pose_dim
        if self.use_pose:
            self.pose_proj = nn.Sequential(
                nn.Linear(pose_dim, 64),
                nn.ReLU(inplace=True),
                nn.Dropout(dropout),
            )
            classifier_in = hidden_dim + 64
        else:
            self.pose_proj = None
            classifier_in = hidden_dim
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(classifier_in, num_classes),
        )

    def forward(self, x, pose=None):
        batch_size, time_steps, channels, height, width = x.shape
        x = x.reshape(batch_size * time_steps, channels, height, width)
        x = self.backbone(x)
        x = self.pool(x).flatten(1)
        x = self.feature_proj(x)
        x = x.reshape(batch_size, time_steps, -1)
        temporal_out, _ = self.temporal(x)
        temporal_feat = temporal_out[:, -1, :]
        if self.use_pose:
            if pose is None:
                pose = torch.zeros(batch_size, time_steps, self.pose_dim, device=temporal_feat.device)
            pose_feat = self.pose_proj(pose.mean(dim=1))
            temporal_feat = torch.cat([temporal_feat, pose_feat], dim=1)
        logits = self.classifier(temporal_feat)
        return logits


class CricketPoseExtractor:
    def __init__(self, enabled=False):
        self.enabled = bool(enabled and MEDIAPIPE_AVAILABLE)
        self.pose_dim = 132
        self._pose = None
        if self.enabled:
            self._pose = mp.solutions.pose.Pose(
                static_image_mode=False,
                model_complexity=1,
                enable_segmentation=False,
                smooth_landmarks=True,
                min_detection_confidence=0.5,
                min_tracking_confidence=0.5,
            )

    def extract(self, frame_bgr):
        if not self.enabled:
            return np.zeros(self.pose_dim, dtype=np.float32)
        rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        results = self._pose.process(rgb)
        if results.pose_landmarks is None:
            return np.zeros(self.pose_dim, dtype=np.float32)
        values = []
        for landmark in results.pose_landmarks.landmark:
            values.extend([landmark.x, landmark.y, landmark.z, landmark.visibility])
        return np.asarray(values, dtype=np.float32)

    def close(self):
        if self._pose is not None:
            self._pose.close()


In [ ]:
MEAN = torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32).view(1, 1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32).view(1, 1, 3, 1, 1)


def format_timestamp(seconds):
    minutes = int(seconds // 60)
    rem = seconds - minutes * 60
    return f"{minutes:02d}:{rem:05.2f}"


def find_uploaded_model(explicit_path=None):
    if explicit_path and Path(explicit_path).exists():
        return str(Path(explicit_path))
    search_roots = [Path("/kaggle/input"), WORK_DIR, Path.cwd()]
    candidates = []
    for root in search_roots:
        if not root.exists():
            continue
        for pattern in ("*.pth", "*.pt", "*.bin"):
            for candidate in root.rglob(pattern):
                candidate_str = str(candidate)
                if any(part in candidate.parts for part in (".venv", "site-packages", "__pycache__")):
                    continue
                try:
                    if candidate.stat().st_size < 100_000:
                        continue
                except OSError:
                    continue
                candidates.append(candidate)
    if not candidates:
        return None

    def rank_key(path_obj):
        name = path_obj.name.lower()
        score = 0
        score += 8 if "mobile" in name else 0
        score += 6 if "shot" in name or "cricket" in name else 0
        score += 4 if "realtime" in name else 0
        score += 2 if "lstm" in name else 0
        return (-score, len(str(path_obj)))

    candidates = sorted(set(candidates), key=rank_key)
    return str(candidates[0])


def generate_demo_video(output_path=DEMO_VIDEO_PATH, fps=24, seconds=12, width=960, height=540):
    output_path = str(output_path)
    if Path(output_path).exists():
        return output_path
    writer = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height))
    if not writer.isOpened():
        raise RuntimeError("Unable to create demo video")
    total_frames = fps * seconds
    shot_schedule = [
        (0, "Defence"),
        (2, "Cover Drive"),
        (4, "Straight Drive"),
        (6, "Pull Shot"),
        (8, "Cut Shot"),
        (10, "Sweep"),
    ]
    for frame_idx in range(total_frames):
        t = frame_idx / fps
        canvas = np.zeros((height, width, 3), dtype=np.uint8)
        canvas[:] = (28, 90, 38)
        cv2.rectangle(canvas, (0, int(height * 0.68)), (width, height), (70, 150, 220), -1)
        cv2.rectangle(canvas, (int(width * 0.46), int(height * 0.24)), (int(width * 0.54), int(height * 0.68)), (188, 205, 225), 3)
        expected_label = shot_schedule[-1][1]
        for start_time, name in shot_schedule:
            if t >= start_time:
                expected_label = name
        player_x = int(width * 0.50 + 110 * math.sin(2 * math.pi * t / 2.0))
        player_y = int(height * 0.56)
        cv2.circle(canvas, (player_x, player_y - 80), 25, (215, 215, 215), -1)
        cv2.line(canvas, (player_x, player_y - 55), (player_x, player_y + 55), (210, 210, 210), 10)
        cv2.line(canvas, (player_x, player_y - 10), (player_x - 60, player_y + 20), (210, 210, 210), 10)
        cv2.line(canvas, (player_x, player_y - 10), (player_x + 55, player_y + 10), (210, 210, 210), 10)
        cv2.line(canvas, (player_x, player_y + 55), (player_x - 35, player_y + 140), (210, 210, 210), 10)
        cv2.line(canvas, (player_x, player_y + 55), (player_x + 35, player_y + 140), (210, 210, 210), 10)
        bat_angle = -50 + 40 * math.sin(2 * math.pi * t)
        bat_len = 120
        bat_end = (
            int(player_x + bat_len * math.cos(math.radians(bat_angle))),
            int(player_y - 40 + bat_len * math.sin(math.radians(bat_angle))),
        )
        cv2.line(canvas, (player_x + 15, player_y - 10), bat_end, (60, 130, 200), 14)
        ball_x = int(width * 0.12 + (width * 0.72) * ((frame_idx % fps) / fps))
        ball_y = int(height * 0.50 + 35 * math.sin(2 * math.pi * t * 2.0))
        cv2.circle(canvas, (ball_x, ball_y), 12, (25, 25, 220), -1)
        cv2.putText(canvas, "Synthetic demo stream", (30, 45), cv2.FONT_HERSHEY_SIMPLEX, 1.1, (255, 255, 255), 2, cv2.LINE_AA)
        cv2.putText(canvas, f"Expected shot: {expected_label}", (30, 90), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 244, 160), 2, cv2.LINE_AA)
        cv2.putText(canvas, f"Time: {format_timestamp(t)}", (30, 135), cv2.FONT_HERSHEY_SIMPLEX, 0.95, (255, 255, 255), 2, cv2.LINE_AA)
        writer.write(canvas)
    writer.release()
    return output_path


def find_video_source(preferred_source=0, fallback_video_path=None):
    webcam_cap = cv2.VideoCapture(preferred_source)
    if webcam_cap.isOpened():
        ok, frame = webcam_cap.read()
        webcam_cap.release()
        if ok and frame is not None and frame.size > 0:
            return preferred_source, "webcam"
    else:
        webcam_cap.release()

    if fallback_video_path and Path(fallback_video_path).exists():
        return str(fallback_video_path), "video"

    video_exts = {".mp4", ".avi", ".mov", ".mkv", ".webm", ".mpeg"}
    search_roots = [Path("/kaggle/input"), WORK_DIR, Path.cwd()]
    for root in search_roots:
        if not root.exists():
            continue
        candidates = []
        for candidate in root.rglob("*"):
            if candidate.suffix.lower() not in video_exts:
                continue
            if candidate.name in {Path(OUTPUT_VIDEO_PATH).name, Path(DEMO_VIDEO_PATH).name}:
                continue
            candidates.append(candidate)
        if candidates:
            return str(sorted(candidates)[0]), "video"

    return generate_demo_video(), "video"


def extract_state_dict(checkpoint_obj):
    if isinstance(checkpoint_obj, dict):
        for key in ("state_dict", "model_state_dict", "model", "net", "weights"):
            value = checkpoint_obj.get(key)
            if isinstance(value, dict):
                return value
        tensor_values = [value for value in checkpoint_obj.values() if isinstance(value, torch.Tensor)]
        if tensor_values:
            return checkpoint_obj
    if hasattr(checkpoint_obj, "state_dict"):
        return checkpoint_obj.state_dict()
    return checkpoint_obj


def load_model(model_path=None, device=DEVICE, use_pose=USE_POSE):
    resolved_model_path = find_uploaded_model(model_path)
    model = MobileNetV2LSTM(num_classes=len(CLASS_NAMES), use_pose=use_pose)
    metadata = {
        "resolved_model_path": resolved_model_path,
        "loaded_checkpoint": False,
        "message": "Initialized MobileNetV2 + LSTM with random weights",
    }
    if resolved_model_path is not None:
        try:
            checkpoint = torch.load(resolved_model_path, map_location="cpu")
            state_dict = extract_state_dict(checkpoint)
            if not isinstance(state_dict, dict):
                raise RuntimeError("Unsupported checkpoint format")
            current_state = model.state_dict()
            compatible_state = {}
            for key, value in state_dict.items():
                clean_key = key.replace("module.", "")
                if clean_key in current_state and tuple(current_state[clean_key].shape) == tuple(value.shape):
                    compatible_state[clean_key] = value
            if not compatible_state:
                raise RuntimeError("No compatible tensors found for MobileNetV2LSTM")
            model.load_state_dict(compatible_state, strict=False)
            metadata["loaded_checkpoint"] = True
            metadata["message"] = f"Loaded {len(compatible_state)} compatible tensors from {Path(resolved_model_path).name}"
        except Exception as exc:
            metadata["message"] = f"Checkpoint load failed: {exc}"
    model.to(device)
    model.eval()
    return model, metadata


def preprocess_frames(frames, device=DEVICE):
    if len(frames) == 0:
        raise ValueError("Frame buffer is empty")
    array = np.stack(frames).astype(np.float32) / 255.0
    array = array[..., ::-1].copy()
    tensor = torch.from_numpy(array).permute(0, 3, 1, 2).unsqueeze(0)
    tensor = (tensor - MEAN) / STD
    return tensor.to(device, non_blocking=True)


def build_pose_tensor(frames, pose_extractor, device=DEVICE):
    if pose_extractor is None or not pose_extractor.enabled:
        return None
    pose_seq = np.stack([pose_extractor.extract(frame) for frame in frames]).astype(np.float32)
    return torch.from_numpy(pose_seq).unsqueeze(0).to(device, non_blocking=True)


def predict_shot(model, frames, pose_extractor=None, device=DEVICE):
    inputs = preprocess_frames(frames, device=device)
    pose_tensor = build_pose_tensor(frames, pose_extractor, device=device)
    with torch.no_grad():
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            logits = model(inputs, pose_tensor)
            probs = F.softmax(logits, dim=1).squeeze(0).detach().cpu().numpy()
    pred_idx = int(np.argmax(probs))
    confidence = float(probs[pred_idx])
    return CLASS_NAMES[pred_idx], confidence, probs


def smooth_predictions(prob_history, class_names=CLASS_NAMES, threshold=CONF_THRESH):
    if len(prob_history) == 0:
        return "Detecting...", 0.0, None
    avg_probs = np.mean(np.stack(prob_history), axis=0)
    pred_idx = int(np.argmax(avg_probs))
    confidence = float(avg_probs[pred_idx])
    label = class_names[pred_idx] if confidence >= threshold else "Detecting..."
    return label, confidence, avg_probs


def draw_overlay(frame, label, confidence, fps_value, source_kind, threshold, model_message):
    output = frame.copy()
    h, w = output.shape[:2]
    panel_h = 132
    cv2.rectangle(output, (0, 0), (w, panel_h), (18, 18, 18), -1)
    display_label = label if confidence >= threshold else "Detecting..."
    label_color = (60, 220, 60) if display_label != "Detecting..." else (0, 215, 255)
    model_line = model_message if len(model_message) <= 48 else model_message[:45] + "..."
    cv2.putText(output, f"Shot: {display_label}", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.0, label_color, 2, cv2.LINE_AA)
    cv2.putText(output, f"Confidence: {confidence:.2f}", (20, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.85, (255, 255, 255), 2, cv2.LINE_AA)
    cv2.putText(output, f"FPS: {fps_value:.2f}", (20, 118), cv2.FONT_HERSHEY_SIMPLEX, 0.85, (255, 255, 255), 2, cv2.LINE_AA)
    cv2.putText(output, f"Source: {source_kind}", (w - 260, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.85, (255, 255, 255), 2, cv2.LINE_AA)
    cv2.putText(output, model_line, (w - 520, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (200, 200, 200), 2, cv2.LINE_AA)
    cv2.putText(output, "Press q to stop", (w - 220, 118), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (255, 255, 255), 2, cv2.LINE_AA)
    return output


def render_frame(frame, inline=False, frame_idx=0, display_stride=INLINE_DISPLAY_EVERY):
    if inline:
        if frame_idx % max(int(display_stride), 1) != 0:
            return True
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        clear_output(wait=True)
        display(Image.fromarray(rgb))
        return True
    cv2.imshow(WINDOW_NAME, frame)
    return True


def save_logs(shot_log, csv_path=TIMESTAMP_CSV_PATH, json_path=TIMESTAMP_JSON_PATH):
    Path(csv_path).parent.mkdir(parents=True, exist_ok=True)
    records = [
        {
            "timestamp_sec": float(timestamp_sec),
            "shot_label": str(shot_label),
            "confidence": float(confidence),
        }
        for timestamp_sec, shot_label, confidence in shot_log
    ]
    df = pd.DataFrame(records, columns=["timestamp_sec", "shot_label", "confidence"])
    df.to_csv(csv_path, index=False)
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(records, f, indent=2)
    return df


In [ ]:
def main(
    source=VIDEO_SOURCE,
    fallback_video_path=VIDEO_PATH,
    model_path=MODEL_PATH,
    output_path=OUTPUT_VIDEO_PATH,
    sequence_length=SEQ_LEN,
    inference_stride=INFER_EVERY,
    smoothing_window=SMOOTH_WINDOW,
    confidence_threshold=CONF_THRESH,
    use_pose=USE_POSE,
    display_inline=SHOW_INLINE,
    simulate_realtime=SIMULATE_REALTIME_FOR_FILE,
    max_frames=MAX_FRAMES,
    inline_display_stride=INLINE_DISPLAY_EVERY,
):
    output_path = str(output_path)
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)

    model, model_info = load_model(model_path=model_path, device=DEVICE, use_pose=use_pose)
    pose_extractor = CricketPoseExtractor(enabled=use_pose)
    resolved_source, source_kind = find_video_source(preferred_source=source, fallback_video_path=fallback_video_path)
    cap = cv2.VideoCapture(resolved_source)
    if not cap.isOpened():
        pose_extractor.close()
        raise RuntimeError(f"Unable to open source: {resolved_source}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    if not fps or np.isnan(fps) or fps <= 1:
        fps = 24.0
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 1280)
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 720)

    writer = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height))
    if not writer.isOpened():
        cap.release()
        pose_extractor.close()
        raise RuntimeError(f"Unable to create output video: {output_path}")

    frame_buffer = deque(maxlen=sequence_length)
    prob_history = deque(maxlen=smoothing_window)
    shot_log = []
    current_label = "Detecting..."
    current_conf = 0.0
    frame_idx = 0
    fps_value = 0.0
    last_tick = time.time()
    stream_start = last_tick
    next_stream_time = stream_start
    last_logged_label = None
    last_logged_time = -1e9

    try:
        while True:
            ok, frame = cap.read()
            if not ok or frame is None:
                break
            if frame.size == 0:
                continue

            now = time.time()
            frame_interval = max(now - last_tick, 1e-6)
            instant_fps = 1.0 / frame_interval
            fps_value = instant_fps if fps_value == 0 else (0.90 * fps_value + 0.10 * instant_fps)
            last_tick = now
            frame_idx += 1

            resized = cv2.resize(frame, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
            frame_buffer.append(resized)

            if len(frame_buffer) == sequence_length and frame_idx % max(int(inference_stride), 1) == 0:
                try:
                    _, _, probs = predict_shot(model, list(frame_buffer), pose_extractor=pose_extractor, device=DEVICE)
                    prob_history.append(probs)
                    current_label, current_conf, _ = smooth_predictions(prob_history, threshold=confidence_threshold)
                except Exception:
                    current_label, current_conf = "Detecting...", 0.0
                    print(traceback.format_exc())

            if source_kind == "video":
                timestamp_sec = cap.get(cv2.CAP_PROP_POS_MSEC) / 1000.0
                if timestamp_sec <= 0:
                    timestamp_sec = frame_idx / fps
            else:
                timestamp_sec = time.time() - stream_start

            if current_label != "Detecting..." and current_conf >= confidence_threshold:
                if current_label != last_logged_label or (timestamp_sec - last_logged_time) >= LOG_COOLDOWN:
                    shot_log.append((round(float(timestamp_sec), 2), current_label, round(float(current_conf), 4)))
                    last_logged_label = current_label
                    last_logged_time = timestamp_sec

            overlay = draw_overlay(
                frame,
                current_label,
                current_conf,
                fps_value,
                source_kind,
                confidence_threshold,
                model_info["message"],
            )
            cv2.putText(
                overlay,
                f"Time: {format_timestamp(timestamp_sec)}",
                (20, overlay.shape[0] - 24),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.85,
                (255, 255, 255),
                2,
                cv2.LINE_AA,
            )
            writer.write(overlay)
            render_frame(overlay, inline=display_inline, frame_idx=frame_idx, display_stride=inline_display_stride)

            if not display_inline:
                if (cv2.waitKey(1) & 0xFF) == ord("q"):
                    break

            if max_frames is not None and frame_idx >= int(max_frames):
                break

            if source_kind == "video" and simulate_realtime:
                next_stream_time += 1.0 / fps
                sleep_time = next_stream_time - time.time()
                if sleep_time > 0:
                    time.sleep(sleep_time)
    finally:
        cap.release()
        writer.release()
        pose_extractor.close()
        cv2.destroyAllWindows()

    logs_df = save_logs(shot_log)
    return {
        "source": str(resolved_source),
        "source_kind": source_kind,
        "output_video": output_path,
        "timestamp_csv": TIMESTAMP_CSV_PATH,
        "timestamp_json": TIMESTAMP_JSON_PATH,
        "shot_logs": shot_log,
        "logs_df": logs_df,
        "model_info": model_info,
    }


In [ ]:
RUNTIME_CONFIG = {
    "VIDEO_SOURCE": VIDEO_SOURCE,
    "VIDEO_PATH": VIDEO_PATH,
    "MODEL_PATH": MODEL_PATH,
    "USE_POSE": USE_POSE and MEDIAPIPE_AVAILABLE,
    "SHOW_INLINE": SHOW_INLINE,
    "SEQ_LEN": SEQ_LEN,
    "INFER_EVERY": INFER_EVERY,
    "SMOOTH_WINDOW": SMOOTH_WINDOW,
    "CONF_THRESH": CONF_THRESH,
    "MAX_FRAMES": MAX_FRAMES,
    "SIMULATE_REALTIME": SIMULATE_REALTIME_FOR_FILE,
}
RUNTIME_CONFIG


In [ ]:
results = main(
    source=RUNTIME_CONFIG["VIDEO_SOURCE"],
    fallback_video_path=RUNTIME_CONFIG["VIDEO_PATH"],
    model_path=RUNTIME_CONFIG["MODEL_PATH"],
    output_path=OUTPUT_VIDEO_PATH,
    sequence_length=RUNTIME_CONFIG["SEQ_LEN"],
    inference_stride=RUNTIME_CONFIG["INFER_EVERY"],
    smoothing_window=RUNTIME_CONFIG["SMOOTH_WINDOW"],
    confidence_threshold=RUNTIME_CONFIG["CONF_THRESH"],
    use_pose=RUNTIME_CONFIG["USE_POSE"],
    display_inline=RUNTIME_CONFIG["SHOW_INLINE"],
    simulate_realtime=RUNTIME_CONFIG["SIMULATE_REALTIME"],
    max_frames=RUNTIME_CONFIG["MAX_FRAMES"],
)

summary = {
    "source": results["source"],
    "source_kind": results["source_kind"],
    "output_video": results["output_video"],
    "timestamp_csv": results["timestamp_csv"],
    "timestamp_json": results["timestamp_json"],
    "model_info": results["model_info"],
    "num_logged_events": len(results["shot_logs"]),
    "logged_events_preview": results["shot_logs"][:10],
}
print(json.dumps(summary, indent=2))
display(results["logs_df"].head(20))

if Video is not None and Path(results["output_video"]).exists():
    display(Video(results["output_video"], embed=True, html_attributes="controls autoplay loop muted"))


In [ ]:
def gradio_infer(video_path, use_pose=False, threshold=0.60, seq_len=16, infer_every=5):
    if isinstance(video_path, dict):
        video_path = video_path.get("path") or video_path.get("name")
    if video_path is None or not Path(video_path).exists():
        return None, pd.DataFrame(columns=["timestamp_sec", "shot_label", "confidence"])

    gradio_output = str(WORK_DIR / "gradio_realtime_output.mp4")
    gradio_csv = str(WORK_DIR / "gradio_shot_timestamps.csv")
    gradio_json = str(WORK_DIR / "gradio_shot_timestamps.json")
    global OUTPUT_VIDEO_PATH, TIMESTAMP_CSV_PATH, TIMESTAMP_JSON_PATH
    old_output, old_csv, old_json = OUTPUT_VIDEO_PATH, TIMESTAMP_CSV_PATH, TIMESTAMP_JSON_PATH
    OUTPUT_VIDEO_PATH, TIMESTAMP_CSV_PATH, TIMESTAMP_JSON_PATH = gradio_output, gradio_csv, gradio_json
    try:
        res = main(
            source=video_path,
            fallback_video_path=video_path,
            model_path=MODEL_PATH,
            output_path=gradio_output,
            sequence_length=int(seq_len),
            inference_stride=int(infer_every),
            smoothing_window=SMOOTH_WINDOW,
            confidence_threshold=float(threshold),
            use_pose=bool(use_pose),
            display_inline=False,
            simulate_realtime=False,
            max_frames=None,
        )
        return res["output_video"], res["logs_df"]
    finally:
        OUTPUT_VIDEO_PATH, TIMESTAMP_CSV_PATH, TIMESTAMP_JSON_PATH = old_output, old_csv, old_json


if LAUNCH_GRADIO and GRADIO_AVAILABLE:
    demo = gr.Interface(
        fn=gradio_infer,
        inputs=[
            gr.Video(label="Upload cricket video", sources=["upload"]),
            gr.Checkbox(value=USE_POSE and MEDIAPIPE_AVAILABLE, label="Enable MediaPipe pose"),
            gr.Slider(0.30, 0.95, value=CONF_THRESH, step=0.05, label="Confidence threshold"),
            gr.Slider(8, 32, value=SEQ_LEN, step=4, label="Sequence length"),
            gr.Slider(1, 10, value=INFER_EVERY, step=1, label="Inference stride"),
        ],
        outputs=[
            gr.Video(label="Processed output"),
            gr.Dataframe(label="Shot timestamps"),
        ],
        title="Real-Time Cricket Shot Detection",
        allow_flagging="never",
    )
    demo.launch(inline=True, share=False)
else:
    print("Set LAUNCH_GRADIO = True to launch the Gradio demo when gradio is available.")
